# Fake News Detection - End-to-End NLP Project

**Team:** Sabeur, Philippe, Joao  
**Duration:** 3 days  
**Goal:** Build an accurate, reliable, and reproducible Fake News Detection system.

> **Task:** Binary text classification on news headlines.  
> `0 = Fake`, `1 = Real`

---

## Team Work Organisation

| # | Member | Role | Main Deliverable |
|---|--------|------|------------------|
| 1 | **Sabeur** | Data & Preprocessing Lead | `clean_text(text) -> processed_text` |
| 2 | **Philippe** | Feature Engineering & Modeling Lead | `model_pipeline.fit(X_train, y_train)` |
| 3 | **Joao** | Evaluation, Testing & Deployment Lead | `predictions = model.predict(X_test)` |

### 3-Day Timeline

**Day 1 - Build the Foundation (Sabeur + Philippe)**
- 09:00-12:00 - Sabeur - Load data, fix issues, EDA
- 13:00-17:00 - Sabeur - Build & test preprocessing pipeline
- 17:00-18:00 - Philippe - Set up TF-IDF, baseline model, quick check

**Day 2 - Modeling & Evaluation (Philippe + Joao)**
- 09:00-12:00 - Philippe - Train multiple models (LR, NB, SVM)
- 13:00-17:00 - Philippe - Hyperparameter tuning, cross-validation
- 17:00-18:00 - Joao - Train/validation split, evaluation framework

**Day 3 - Testing & Final Output (Joao + Team)**
- 09:00-12:00 - Joao - Run predictions on `test.csv`
- 13:00-15:30 - Joao - Error analysis & insights
- 15:30-18:00 - Team - Review results, finalize submission, presentation

### Integration Rules (Important)
- Sabeur defines `clean_text()` -> **Philippe MUST use it** (no custom preprocessing)
- Philippe defines `vectorizer + model` -> **Joao MUST reuse it** (no retraining)
- Do **NOT** call `.fit()` / `.fit_transform()` on test data -> use only `.transform()` and `.predict()`

### End-to-End Workflow

```
Raw Data --> Data Cleaning --> Preprocessing --> Feature Engineering --> Model Training --> Evaluation --> Predictions
 (csv)        (Sabeur)         (Sabeur)           (Philippe)             (Philippe)        (Joao)        (Joao)
```

---
## Setup & Imports

All shared libraries used across the pipeline. Each member adds the imports they need in their own section, but core ones are loaded here once.

**Tools & libraries:** `pandas`, `numpy`, `scikit-learn`, `nltk`, `matplotlib`, `seaborn`, `joblib`.

In [ ]:
# Core
import re
import string
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ML
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Persistence
import joblib

# NLTK resources (run once)
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('punkt')

RANDOM_STATE = 42
pd.set_option('display.max_colwidth', 120)

---
# 1. Data Sources

Two CSV files in the `dataset/` folder:

| File | Columns | Notes |
|------|---------|-------|
| `training_data_lowercase.csv` | `label`, `headline` | `0 = Fake`, `1 = Real`. Used to train & validate. |
| `testing_data_lowercase_nolabels.csv` | `headline` only | No labels - predictions will be generated. |

**Examples (training):**
- `0` - *donald trump sends out embarrassing...*
- `1` - *pope francis just called out donald trump...*

**Examples (test):**
- *germany's fdp look to fill schaeuble's big shoes...*
- *france's macron says his job not 'cool' cites talks with turkey's erdogan...*

---
# 2. Data Loading & Cleaning  -  Sabeur

**Goal:** Produce a clean, well-formed DataFrame ready for preprocessing.

**Steps:**
1. Load CSV with `pandas` (handle separator / encoding).
2. Inspect shape, dtypes, head/tail.
3. Handle missing values (drop or impute headlines).
4. Remove exact duplicates.
5. Fix columns / formatting (rename, ensure `label` is int, `headline` is str).
6. Lowercase text (already lowercase in source - verify).

**Output:** `df_train` and `df_test` cleaned and consistent.

In [ ]:
# Load training and test data
TRAIN_PATH = 'dataset/training_data_lowercase.csv'
TEST_PATH  = 'dataset/testing_data_lowercase_nolabels.csv'

# TODO: confirm the separator (\t or , or ;) and adjust
df_train = pd.read_csv(TRAIN_PATH, sep='\t', header=None, names=['label', 'headline'])
df_test  = pd.read_csv(TEST_PATH,  sep='\t', header=None, names=['headline'])

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)
df_train.head()

In [ ]:
# Missing values
print('Missing in train:\n', df_train.isna().sum())
print('\nMissing in test:\n',  df_test.isna().sum())

# Duplicates
n_dup = df_train.duplicated().sum()
print(f'\nDuplicates in train: {n_dup}')
df_train = df_train.drop_duplicates().reset_index(drop=True)

# Type enforcement
df_train['label'] = df_train['label'].astype(int)
df_train['headline'] = df_train['headline'].astype(str)
df_test['headline']  = df_test['headline'].astype(str)

df_train.info()

## 2.1 Exploratory Data Analysis (EDA)  -  Sabeur

Quick look at:
- **Class balance** - Fake vs Real distribution.
- **Headline length** - characters / words per class.
- **Most frequent words** per class (before/after stopword removal).
- **Word clouds** (optional).

Goal: spot biases, leakage, or class imbalance early so Philippe and Joao can plan accordingly.

In [ ]:
# Class balance
ax = df_train['label'].value_counts().plot(kind='bar', color=['#e74c3c', '#2ecc71'])
ax.set_xticklabels(['Fake (0)', 'Real (1)'], rotation=0)
ax.set_title('Class distribution')
plt.show()

# Headline length
df_train['n_words'] = df_train['headline'].str.split().str.len()
sns.histplot(data=df_train, x='n_words', hue='label', bins=40, kde=True)
plt.title('Headline length (words) by class')
plt.show()

---
# 3. NLP Preprocessing  -  Sabeur

**Deliverable:** a single function `clean_text(text) -> processed_text` defined in this notebook.
Philippe and Joao MUST reuse this exact function - no custom preprocessing on their side.

**Pipeline steps:**
1. **Tokenization** - split text into tokens.
   *e.g. `"this is great"` -> `["this", "is", "great"]`*
2. **Stopword removal** - drop common English stopwords.
   *e.g. `["this", "is", "great"]` -> `["great"]`*
3. **Punctuation removal** - strip punctuation & special characters.
   *e.g. `"great!!!"` -> `"great"`*
4. **Stemming OR Lemmatization** - reduce words to their base form.
   *e.g. `running -> run`, `better -> good`*
   Pick **one** (we use lemmatization here for readability).

**Output:** a `processed_text` column on both train and test DataFrames, ready for vectorization in section 4.

In [ ]:
STOPWORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
PUNCT_RE   = re.compile(f'[{re.escape(string.punctuation)}0-9]')

def clean_text(text: str) -> str:
    """Sabeur's canonical preprocessing - used by ALL teammates."""
    text = text.lower()
    text = PUNCT_RE.sub(' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Quick sanity check
clean_text("Donald Trump sends out EMBARRASSING tweets!!! #fakenews")

In [ ]:
df_train['processed_text'] = df_train['headline'].apply(clean_text)
df_test['processed_text']  = df_test['headline'].apply(clean_text)

df_train[['headline', 'processed_text', 'label']].head()

---
# 4. Feature Engineering  -  Philippe

**Goal:** Turn `processed_text` into a numerical feature matrix `X`.

**Approaches:**
- **Bag of Words (BoW)** - raw word counts (`CountVectorizer`).
- **TF-IDF (baseline)** - weights words by importance: `TF x IDF` (`TfidfVectorizer`).
- **n-grams (optional)** - capture short phrases (`ngram_range=(1,2)`).
- **Embeddings (optional improvement)** - Word2Vec, GloVe, BERT.

**Rule:** the vectorizer is fit **only on training data**, then `.transform()` is applied to validation and test.

> *Code to be implemented by Philippe.*

In [ ]:
# Philippe - add your code here


---
# 5. Model Training  -  Philippe

**Models to compare:**

| Family | Model | Notes |
|--------|-------|-------|
| Linear (baseline) | Logistic Regression | Strong baseline for text |
| Probabilistic | Multinomial Naive Bayes | Fast, good with sparse counts |
| Margin | Linear SVM | Often best on TF-IDF |
| *(optional)* | Random Forest / XGBoost | Heavier, ensemble |

**Process:**
1. Train/validation split (stratified).
2. Wrap `vectorizer + model` in a single `sklearn.Pipeline` - guarantees no leakage.
3. Hyperparameter tuning with `GridSearchCV` (cross-validated).
4. Persist the best pipeline with `joblib.dump`.

**Deliverable:** `model_pipeline` fitted on the full training set, ready for Joao.

> *Code to be implemented by Philippe.*

In [ ]:
# Philippe - add your code here


---
# 6. Evaluation  -  Joao

**Goal:** Measure how well the model generalises on the validation set, then run error analysis.

**Metrics:**
- **Accuracy** - overall correctness (only meaningful if classes are balanced).
- **Precision** - of predicted positives, how many are correct.
- **Recall** - of actual positives, how many we caught.
- **F1-score** - harmonic mean of precision & recall (preferred summary).
- **Confusion matrix** - TP / FP / FN / TN breakdown.

Don't rely only on accuracy, especially with imbalanced data.

**Error analysis:** look at misclassified headlines, identify patterns (length, topic, named entities) and feed insights back to Sabeur / Philippe.

> *Code to be implemented by Joao.*

In [ ]:
# Joao - add your code here


---
# 7. Prediction on Test Data  -  Joao

**Strict rules:**
- Use the **same** `clean_text()` defined by Sabeur.
- Use the **same** fitted vectorizer + model from Philippe - wrapped inside `model_pipeline`.
- Only call `.transform()` and `.predict()` on the test data - **never** `.fit()`.

Workflow:
1. Load `testing_data_lowercase_nolabels.csv`.
2. Apply `clean_text()` to each headline.
3. `predictions = model_pipeline.predict(processed_test)`.

> *Code to be implemented by Joao.*

In [ ]:
# Joao - add your code here


---
# 8. Output / Submission  -  Joao

**Final deliverables:**
1. `submission.csv` - one column `prediction` with `0`/`1` per test headline (in original order).
2. Persisted model: `model_pipeline.joblib`.
3. Evaluation report (metrics + confusion matrix + error-analysis insights).

> *Code to be implemented by Joao.*

In [ ]:
# Joao - add your code here


---
# 9. Conclusion & Next Steps

**What we built:**
- Reproducible NLP pipeline: clean -> vectorize -> classify -> predict.
- Compared LR / NB / SVM on TF-IDF features.
- Generated a labelled `submission.csv` for the unlabelled test set.

**Optional improvements (if time):**
- Word embeddings (Word2Vec / GloVe).
- Transformer-based models (BERT / RoBERTa).
- Simple web app (Streamlit) to demo the classifier live.
- Monitoring: track per-topic accuracy, drift over time.

**Teamwork makes the model work!**